# Llama 3.1 8B Calibrated Listwise Ranking — Enhancement 6a
Combined Logit Margin + Entropy confidence with temperature scaling.

In [ ]:
# CELL 1: Install dependencies
!pip install -q accelerate peft
!pip install -q --upgrade transformers
!pip install -U bitsandbytes>=0.46.1
print('Done!')

In [ ]:
# CELL 2: Mount Drive and set paths

from google.colab import drive
drive.mount('/content/drive')

MODEL_NAME = 'Llama3.1-8B'

BASE_PATH    = '/content/drive/MyDrive/Colab Notebooks/Ranking_Selection/Dataset/'
RESULTS_PATH = f'/content/drive/MyDrive/Colab Notebooks/Ranking_Selection/exp_ Calibrated and Adaptive Listwise Listwise Ranking/{MODEL_NAME}/'

import os
os.makedirs(RESULTS_PATH, exist_ok=True)

In [ ]:
# CELL 3: Imports and config
import pandas as pd
import numpy as np
import torch
import os, re, random, subprocess, sys

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import snapshot_download
from scipy.optimize import minimize_scalar
from scipy.special import expit
from scipy.stats import spearmanr
from sklearn.metrics import ndcg_score
import warnings
warnings.filterwarnings('ignore')

SEED = 42
HF_TOKEN = 'your_huggingface_token_here'

MODEL_IDS = {
    'Mistral-7B':  'mistralai/Mistral-7B-Instruct-v0.3',
    'Llama3.1-8B': 'meta-llama/Llama-3.1-8B-Instruct',
    'Qwen2.5-7B':  'Qwen/Qwen2.5-7B-Instruct',
}

ADAPTER_HF_IDS = {
    'Mistral-7B':  'NajatAlsa/mistral-scheme-a',
    'Llama3.1-8B': 'NajatAlsa/llama-scheme-a',
    'Qwen2.5-7B':  'NajatAlsa/qwen-scheme-a',
}

HF_MODEL_ID   = MODEL_IDS[MODEL_NAME]
ADAPTER_HF_ID = ADAPTER_HF_IDS[MODEL_NAME]
ADAPTER_LOCAL = f'/tmp/calib_adapter_{MODEL_NAME}'
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print(f'Model:  {MODEL_NAME}')
print(f'Device: {DEVICE}')

In [ ]:
# CELL 4: Load model and tokenizer
from huggingface_hub import snapshot_download

sys.path.append('/content/drive/MyDrive/Colab Notebooks/Ranking_Selection/')
from prompt_config import SYSTEM_INSTRUCTION, build_user_content, build_assistant_output

# Download adapter from HuggingFace
ADAPTER_LOCAL = snapshot_download(repo_id=ADAPTER_HF_ID, token=HF_TOKEN)
print(f'Adapter downloaded to {ADAPTER_LOCAL}')

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID, token=HF_TOKEN)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Base model in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
base_model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
    attn_implementation='eager'
)
# Load adapter
model = PeftModel.from_pretrained(base_model, ADAPTER_LOCAL)
model.eval()
print('Model + adapter loaded successfully')

In [ ]:
# CELL 5: Load datasets
val_df  = pd.read_csv(BASE_PATH + 'Scheme_A/obj3_val_A.csv')
test_df = pd.read_csv(BASE_PATH + 'Scheme_A/obj3_test_A.csv')

val_list_ids  = val_df['list_id'].unique().tolist()
test_list_ids = test_df['list_id'].unique().tolist()

print(f'Val lists:  {len(val_list_ids)}')
print(f'Test lists: {len(test_list_ids)}')

In [ ]:
# CELL 6: Prompt builder — uses prompt_config.py (same as SFT evaluation)
def build_prompt(group_df):
    messages = [
        {'role': 'system', 'content': SYSTEM_INSTRUCTION},
        {'role': 'user',   'content': build_user_content(group_df)}
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

print('Prompt builder defined.')

In [ ]:
# CELL 7: Inference with logprobs for confidence computation
def run_inference_with_logprobs(group_df, max_new_tokens=200):
    group = group_df.sample(frac=1, random_state=int(group_df['list_id'].iloc[0])).reset_index(drop=True)
    prompt    = build_prompt(group)
    inputs    = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            return_dict_in_generate=True,
            output_scores=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs.sequences[0][input_len:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    scores = outputs.scores  # tuple of tensors, one per generated token

    return generated_text, scores, group


def compute_confidence(scores, group_df):
    display_ids = group_df['display_id'].tolist()

    # Get token IDs for each node's number suffix (e.g. "5" from "Node_5")
    node_token_ids = []
    for did in display_ids:
        num = did.split('_')[1]
        tids = tokenizer.encode(num, add_special_tokens=False)
        if tids:
            node_token_ids.append(tids[0])

    margins   = []
    entropies = []

    for step_scores in scores:
        probs = torch.softmax(step_scores[0], dim=-1)

        # Logit margin at node positions only
        node_probs = probs[node_token_ids]
        if len(node_probs) >= 2:
            sorted_p = torch.sort(node_probs, descending=True).values
            margins.append((sorted_p[0] - sorted_p[1]).item())

        # Normalised entropy over full vocab
        log_probs = torch.log(probs + 1e-10)
        entropy   = -torch.sum(probs * log_probs).item()
        max_entropy = np.log(len(probs))
        entropies.append(1.0 - entropy / max_entropy)  # higher = more confident

    margin_score  = float(np.mean(margins))   if margins   else 0.0
    entropy_score = float(np.mean(entropies)) if entropies else 0.0

    # Geometric mean of both signals
    confidence = float(np.sqrt(margin_score * entropy_score)) if (margin_score > 0 and entropy_score > 0) else 0.0
    return confidence


def parse_ranking(generated_text, group_df):
    display_ids = group_df['display_id'].tolist()
    lines       = generated_text.strip().split('\n')
    pred_ranks  = {}
    rank        = 1
    for line in lines:
        line = line.strip()
        for did in display_ids:
            if re.search(r'\b' + re.escape(did) + r'\b', line) and did not in pred_ranks:
                pred_ranks[did] = rank
                rank += 1
                break
    for did in display_ids:
        if did not in pred_ranks:
            pred_ranks[did] = rank
            rank += 1
    return pred_ranks


def compute_top1(group_df, pred_ranks):
    true_top1 = group_df[group_df['list_rank'] == 1]['display_id'].values[0]
    pred_top1 = min(pred_ranks, key=pred_ranks.get)
    return int(true_top1 == pred_top1)

print('Functions defined.')

In [ ]:
# CELL 8: TOPSIS fallback — fixed with shuffling to prevent index-0 bias

NORM_COLS = ['Response_Time_norm', 'Availability_norm', 'Throughput_norm',
             'Reliability_norm', 'Latency_norm']

def run_topsis(group_df):
    # Shuffle to prevent ground-truth winner always being at index 0
    group_df = group_df.sample(frac=1, random_state=int(group_df['list_id'].iloc[0])).reset_index(drop=True)
    w = np.array([0.2, 0.2, 0.2, 0.2, 0.2])
    matrix      = group_df[NORM_COLS].values.astype(float)
    weighted    = matrix * w
    ideal_best  = weighted.max(axis=0)
    ideal_worst = weighted.min(axis=0)
    dist_best   = np.sqrt(((weighted - ideal_best)  ** 2).sum(axis=1))
    dist_worst  = np.sqrt(((weighted - ideal_worst) ** 2).sum(axis=1))
    closeness   = dist_worst / (dist_best + dist_worst + 1e-10)
    topsis_top1_idx = np.argmax(closeness)
    topsis_top1_did = group_df['display_id'].values[topsis_top1_idx]
    true_top1_did   = group_df[group_df['list_rank'] == 1]['display_id'].values[0]
    return int(topsis_top1_did == true_top1_did)

print('TOPSIS fallback defined with shuffle fix.')

In [ ]:
# CELL 9: Run inference on VALIDATION set
print('Running inference on validation set...')
val_results = []
val_partial_path = RESULTS_PATH + 'val_confidence_results_partial.csv'

if os.path.exists(val_partial_path):
    done_df   = pd.read_csv(val_partial_path)
    done_ids  = set(done_df['list_id'].tolist())
    val_results = done_df.to_dict('records')
    print(f'Resuming val from checkpoint: {len(done_ids)} lists done')
else:
    done_ids = set()
    print('Starting val fresh')

val_list_ids = val_df['list_id'].unique().tolist()
remaining    = [lid for lid in val_list_ids if lid not in done_ids]
print(f'Val remaining: {len(remaining)} lists')

for i, lid in enumerate(remaining):
    if i % 10 == 0:
        print(f'  Val list {i}/{len(remaining)}...')
    group = val_df[val_df['list_id'] == lid].copy()
    generated_text, scores, group_shuffled = run_inference_with_logprobs(group)
    confidence = compute_confidence(scores, group_shuffled)
    pred_ranks = parse_ranking(generated_text, group_shuffled)
    top1       = compute_top1(group_shuffled, pred_ranks)
    val_results.append({
        'list_id':    lid,
        'confidence': confidence,
        'top1':       top1,
        'generated':  generated_text
    })
    if (i + 1) % 5 == 0:
        pd.DataFrame(val_results).to_csv(val_partial_path, index=False)
        print(f'  Val checkpoint saved: {len(val_results)} lists')

val_results_df = pd.DataFrame(val_results)
val_results_df.to_csv(RESULTS_PATH + 'val_confidence_results.csv', index=False)
print(f'Val done. Mean confidence: {val_results_df["confidence"].mean():.4f}')
print(f'Val Top-1: {val_results_df["top1"].mean()*100:.2f}%')

In [ ]:
# Load saved validation results
val_results_df = pd.read_csv(RESULTS_PATH + 'val_confidence_results.csv')
print(f'Loaded val results: {len(val_results_df)} lists')
print(f'Val Top-1: {val_results_df["top1"].mean()*100:.2f}%')
print(f'Val Mean confidence: {val_results_df["confidence"].mean():.4f}')

In [ ]:
# CELL 10: Temperature scaling calibration on validation set
from scipy.optimize import minimize_scalar

raw_confidences = val_results_df['confidence'].values
true_labels     = val_results_df['top1'].values

def calibration_loss(temperature):
    eps = 1e-6
    log_odds = np.log(np.clip(raw_confidences, eps, 1-eps) /
                      (1 - np.clip(raw_confidences, eps, 1-eps)))
    scaled_probs = expit(log_odds / temperature)
    nll = -np.mean(
        true_labels * np.log(scaled_probs + eps) +
        (1 - true_labels) * np.log(1 - scaled_probs + eps)
    )
    return nll

result = minimize_scalar(calibration_loss, bounds=(0.1, 10.0), method='bounded')
OPTIMAL_TEMPERATURE = result.x
print(f'Optimal temperature: {OPTIMAL_TEMPERATURE:.4f}')

def apply_temperature_scaling(confidence, temperature):
    eps = 1e-6
    log_odds = np.log(np.clip(confidence, eps, 1-eps) /
                      (1 - np.clip(confidence, eps, 1-eps)))
    return float(expit(log_odds / temperature))

print('Temperature scaling calibrated.')

In [ ]:
# CELL 11: Run inference on TEST set
partial_path = RESULTS_PATH + 'test_confidence_results_partial.csv'
full_path    = RESULTS_PATH + 'test_confidence_results.csv'

done_ids     = set()
test_results = []

if os.path.exists(partial_path):
    done_df      = pd.read_csv(partial_path)
    done_ids     = set(done_df['list_id'].tolist())
    test_results = done_df.to_dict('records')
    print(f'Resuming from checkpoint: {len(done_ids)} lists already done')

test_list_ids = test_df['list_id'].unique().tolist()
remaining     = [lid for lid in test_list_ids if lid not in done_ids]
print(f'Remaining: {len(remaining)} lists to process')

for i, lid in enumerate(remaining):
    if i % 10 == 0:
        print(f'  List {i}/{len(remaining)}...')

    base_group = test_df[test_df['list_id'] == lid].copy().reset_index(drop=True)

    llm_group = base_group.copy()
    generated_text, scores, group_shuffled = run_inference_with_logprobs(llm_group)
    raw_confidence = compute_confidence(scores, group_shuffled)
    cal_confidence = apply_temperature_scaling(raw_confidence, OPTIMAL_TEMPERATURE)
    pred_ranks     = parse_ranking(generated_text, group_shuffled)
    llm_top1       = compute_top1(group_shuffled, pred_ranks)

    topsis_group = base_group.copy()
    topsis_top1  = run_topsis(topsis_group)

    test_results.append({
        'list_id':        lid,
        'raw_confidence': raw_confidence,
        'cal_confidence': cal_confidence,
        'llm_top1':       llm_top1,
        'topsis_top1':    topsis_top1,
        'generated':      generated_text
    })

    if (i + 1) % 5 == 0:
        pd.DataFrame(test_results).to_csv(partial_path, index=False)
        print(f'  Checkpoint saved: {len(test_results)} lists total')

test_results_df = pd.DataFrame(test_results)
test_results_df.to_csv(full_path, index=False)
print(f'\nDone! {len(test_results_df)} lists processed')
print(f'LLM Top-1:    {test_results_df["llm_top1"].mean()*100:.2f}%')
print(f'TOPSIS Top-1: {test_results_df["topsis_top1"].mean()*100:.2f}%')
print(f'\nCalibrated confidence stats:')
print(test_results_df['cal_confidence'].describe().round(4))
print(f'\nLists below each threshold:')
for t in [0.5, 0.6, 0.7, 0.8, 0.9]:
    below = (test_results_df['cal_confidence'] < t).sum()
    print(f'  Below {t}: {below} lists ({below/len(test_results_df)*100:.1f}%)')

In [ ]:
# CELL 12: Confidence-gated selection
THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9]
gate_results = []

print('Confidence-gated selection results:')
print(f'{"T":<6} {"Coverage(%)":<15} {"Accepted(n)":<15} {"Accuracy(%)":<15} {"Fallback(n)":<15} {"Combined(%)":<15}')
print('='*80)

for T in THRESHOLDS:
    accepted = test_results_df[test_results_df['cal_confidence'] >= T]
    rejected = test_results_df[test_results_df['cal_confidence'] <  T]

    n_accepted  = len(accepted)
    n_rejected  = len(rejected)
    n_total     = len(test_results_df)
    coverage    = round(n_accepted / n_total * 100, 2)
    acc_accepted = round(accepted['llm_top1'].mean() * 100, 2) if n_accepted > 0 else 0.0
    correct_accepted = accepted['llm_top1'].sum()
    correct_rejected = rejected['topsis_top1'].sum()
    combined_acc = round((correct_accepted + correct_rejected) / n_total * 100, 2)

    print(f'{T:<6} {coverage:<15} {n_accepted:<15} {acc_accepted:<15} {n_rejected:<15} {combined_acc:<15}')
    gate_results.append({
        'T': T, 'coverage': coverage, 'n_accepted': n_accepted,
        'n_rejected': n_rejected, 'acc_accepted': acc_accepted,
        'combined_acc': combined_acc
    })

gate_df = pd.DataFrame(gate_results)
gate_df.to_csv(RESULTS_PATH + 'confidence_gating_results.csv', index=False)
print('\nSaved: confidence_gating_results.csv')

In [ ]:
# CELL 14: Summary

print(f'Optimal temperature:       {OPTIMAL_TEMPERATURE:.4f}')
print(f'LLM baseline Top-1:        {test_results_df["llm_top1"].mean()*100:.2f}%')
print(f'TOPSIS baseline Top-1:     {test_results_df["topsis_top1"].mean()*100:.2f}%')
print()
print('Confidence-gated selection:')
print(gate_df.to_string(index=False))
print()
print('Files saved to:', RESULTS_PATH)